# ДЗ-1 — версия 2: уточнённые аналитические расчёты

Загрузите **`hw1_version_2.ipynb`** в Google Colab, выберите **T4 GPU** и выполните **Runtime → Run all**. Notebook автономен: код актуальных `.py`-модулей встроен. Для первой версии и исходных формул сохранён отдельный `hw1.ipynb`.

Полный эксперимент включает 132 конфигурации, исходную сеть, FP32, фиксированные backend flags и seed 2026. Версия 2 использует два compute rate для latency, active power для energy, округление allocations и документированный постоянный workspace cuBLAS для memory. Память не калибруется по CSV.

Сохранённый отчёт текущей версии находится в `hw1/results_v2/`, первой — в `hw1/results_v1/`. Каждый новый GPU-прогон создаёт отдельную папку **`hw1/results_v2/runs/<RUN_ID>/`**; из папки `hw1` путь начинается с `results_v2/`. В конце скачивается архив только нового прогона. Повтор ячейки измерений продолжает его CSV. После перезапуска runtime задайте прежний RUN_ID и восстановите файлы для продолжения.



In [ ]:
from pathlib import Path
import sys, json, platform, importlib.util, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

for package, module in [('nvidia-ml-py', 'pynvml'), ('scipy', 'scipy')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU runtime в Colab: Runtime → Change runtime type → T4 GPU')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
GPU_NAME = torch.cuda.get_device_name(0)
if 'T4' not in GPU_NAME:
    raise RuntimeError(f'Для этой серии выбран T4; текущая GPU: {GPU_NAME}')
ROOT = Path.cwd().resolve()
from datetime import datetime, timezone
from uuid import uuid4
# Для возобновления после перезапуска можно задать RUN_ID = 'прежний идентификатор'.
RUN_ID = globals().get('RUN_ID') or (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8])
HW1_ROOT = ROOT if ROOT.name == 'hw1' else ROOT / 'hw1'
RUNS_ROOT = HW1_ROOT / 'results_v2' / 'runs'
RESULTS = RUNS_ROOT / RUN_ID
(RESULTS / 'figures').mkdir(parents=True, exist_ok=True)
print('Python:', platform.python_version(), 'PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda, '| GPU:', GPU_NAME)
print('RUN_ID:', RUN_ID)
print('Results:', RESULTS)


## Встроенный код: model, equations, measurement, calibration

Эти ячейки повторяют соответствующие `.py` файлы репозитория, чтобы Colab не зависел от локального файлового дерева.

In [ ]:
"""The exact FP32 inference network used throughout Homework 1."""
from torch import nn


class SmallCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        layers = []
        convs = (
            (3, 32, 7, 2),
            (32, 64, 5, 1),
            (64, 128, 3, 2),
            (128, 256, 1, 1),
            (256, 256, 3, 2),
            (256, 512, 1, 1),
        )
        for index, (in_channels, out_channels, kernel, stride) in enumerate(convs):
            layers += [
                nn.Conv2d(in_channels, out_channels, kernel, stride,
                          padding=kernel // 2, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            ]
            if index == 0:
                layers.append(nn.MaxPool2d(3, stride=2, padding=1))
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 100),
        )

    def forward(self, x):
        return self.head(self.features(x))


In [ ]:
"""Explicit analytic equations. S and B are NumPy-broadcastable scalars/arrays.

Assumptions: NCHW FP32, eval+inference_mode, Conv-BN-ReLU after each conv,
no residuals, inplace ReLU. Arithmetic counts floating add/multiply/divide;
comparisons (ReLU/MaxPool) are not FLOPs. Memory includes live tensors,
512-byte native-allocator rounding of model storage and one persistent
pre-Hopper cuBLAS workspace after warmup. Additional cuDNN/cuBLASLt
workspaces and allocator block reuse are excluded; CUDA context is not
part of torch.cuda.max_memory_allocated().
"""
import numpy as np

# input channels, output channels, kernel, output resolution divisor
CONVS = ((3, 32, 7, 2), (32, 64, 5, 4), (64, 128, 3, 8),
         (128, 256, 1, 8), (256, 256, 3, 16), (256, 512, 1, 16))


def _sb(image_size, batch):
    s, b = np.broadcast_arrays(np.asarray(image_size, dtype=float),
                               np.asarray(batch, dtype=float))
    if np.any(s <= 0) or np.any(b <= 0):
        raise ValueError("S and B must be positive")
    return s, b


def model_bytes():
    """FP32 parameters and BN buffers, including six int64 batch counters."""
    conv_params = sum(ci * co * k * k for ci, co, k, _ in CONVS)
    bn_floats = sum(4 * co for _, co, _, _ in CONVS)
    linear_params = 512 * 256 + 256 + 256 * 100 + 100
    return 4 * (conv_params + bn_floats + linear_params) + 8 * len(CONVS)


def flops(image_size, batch):
    """Forward FLOPs: 2 per MAC, 2 per BN output, 1 per GAP input, biases."""
    s, b = _sb(image_size, batch)
    total = 0.0
    for ci, co, k, divisor in CONVS:
        pixels = (s / divisor) ** 2
        total += b * co * pixels * (2 * ci * k * k + 2)
    total += b * 512 * (s / 16) ** 2  # GAP: (n-1) adds + division
    total += b * (2 * 512 * 256 + 256 + 2 * 256 * 100 + 100)
    return total


def model_allocated_bytes():
    """Model storage with PyTorch native allocator's 512-byte granularity.

    Each parameter/buffer has its own allocation, including six 8-byte BN
    counters. This is derived from tensor shapes, not fitted to measurements.
    """
    sizes = []
    for ci, co, k, _ in CONVS:
        sizes.extend([4 * ci * co * k * k, *([4 * co] * 4), 8])
    sizes.extend([4 * 512 * 256, 4 * 256, 4 * 256 * 100, 4 * 100])
    return sum(512 * ((size + 511) // 512) for size in sizes)


def memory_live_tensors(image_size, batch):
    """Original ideal tensor-storage estimate, retained as a comparison."""
    s, b = _sb(image_size, batch)
    return model_bytes() + 76 * b * s**2


def memory(image_size, batch):
    """Peak estimate for warmed-up T4, FP32, one stream, native allocator.

    M = W_alloc + W_cuBLAS + input + two BN1 activations.
    W_cuBLAS = (2*4096 + 8*16) KiB is the PyTorch pre-Hopper default,
    allocated persistently through the CUDA caching allocator after Linear.
    Source: pytorch v2.11.0/aten/src/ATen/cuda/CublasHandlePool.cpp,
    parseChosenWorkspaceSize()/getNewWorkspace(). It is NOT a fitted offset.

    Assumes no CUBLAS_WORKSPACE_CONFIG override, no extra live CUDA tensors,
    and S a multiple of 16 (input/BN1 sizes are already multiples of 512).
    cuDNN scratch space and additional cuBLASLt pools remain unmodelled;
    the prediction is a baseline, not an exact peak or an OOM guarantee.
    """
    s, b = _sb(image_size, batch)
    cublas_workspace = (2 * 4096 + 8 * 16) * 1024
    return model_allocated_bytes() + cublas_workspace + 76 * b * s**2


def bytes_moved(image_size, batch):
    """Approximate global-memory traffic: each op reads inputs and writes outputs.

Cache reuse, temporary workspaces, kernel fusion and allocator traffic are
excluded. Conv weights are counted once per forward, not once per pixel.
"""
    s, b = _sb(image_size, batch)
    total_elements = 0.0
    previous = 3 * b * s**2
    for index, (ci, co, k, divisor) in enumerate(CONVS):
        output = b * co * (s / divisor)**2
        total_elements += previous + ci * co * k * k + output  # conv
        total_elements += 2 * output + 4 * co  # BN read/write + vectors
        total_elements += 2 * output  # inplace ReLU read/write
        previous = output
        if index == 0:
            pooled = b * co * (s / 4)**2
            total_elements += previous + pooled
            previous = pooled
    total_elements += previous + b * 512  # GAP
    total_elements += b * 512 + 512 * 256 + b * 256  # linear1
    total_elements += 2 * b * 256  # inplace ReLU
    total_elements += b * 256 + 256 * 100 + b * 100  # linear2
    return 4 * total_elements


def latency_terms(image_size, batch, theta):
    """Launch, memory and compute times; optional train-selected batch regime.

    Two effective compute rates approximate a change of convolution execution
    regime. They are empirical rates, not GPU hardware peak specifications.
    Old three-parameter dictionaries retain their original interpretation.
    """
    s, b = _sb(image_size, batch)
    rate = theta["compute_flops_per_s"]
    if theta.get("batch_threshold") is not None:
        rate = np.where(b > theta["batch_threshold"],
                        theta["large_batch_compute_flops_per_s"], rate)
    return (np.full(s.shape, theta["launch_seconds"]),
            bytes_moved(s, b) / theta["memory_bytes_per_s"],
            flops(s, b) / rate)


def latency(image_size, batch, theta):
    """Seconds: t0 + max(F/P(B), Q/R), with a train-calibrated P(B)."""
    launch, traffic, compute = latency_terms(image_size, batch, theta)
    return launch + np.maximum(compute, traffic)


def energy(image_size, batch, theta_energy):
    """Whole-GPU joules: idle + active power over predicted time + work terms."""
    return ((theta_energy["idle_watts"] + theta_energy.get("active_watts", 0.0)) *
            latency(image_size, batch, theta_energy["latency"]) +
            theta_energy["fixed_joules"] +
            theta_energy["joules_per_gflop"] * flops(image_size, batch) / 1e9 +
            theta_energy["joules_per_gbyte"] * bytes_moved(image_size, batch) / 1e9)


In [ ]:
"""GPU measurements for the 132-point grid; safe to resume after OOM/interruption."""
import gc
import math
import threading
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


ACCELERATOR_ERROR = getattr(torch, "AcceleratorError", torch.cuda.OutOfMemoryError)

BASE_S = (32, 64, 128, 224, 256, 384, 512)
BASE_B = (1, 2, 4, 8, 16, 32, 64, 128, 256)


def build_grid(seed=2026):
    rng = np.random.default_rng(seed)
    extra_s = rng.choice([v for v in range(32, 513, 16) if v not in BASE_S],
                         size=4, replace=False)
    extra_b = rng.choice([v for v in range(1, 257) if v not in BASE_B],
                         size=3, replace=False)
    points = [(int(s), int(b)) for s in sorted((*BASE_S, *extra_s))
              for b in sorted((*BASE_B, *extra_b))]
    flags = np.zeros(len(points), dtype=bool)
    flags[rng.choice(len(points), size=round(0.2 * len(points)), replace=False)] = True
    return pd.DataFrame({"S": [p[0] for p in points],
                         "B": [p[1] for p in points],
                         "is_validation": flags})


def _synchronize():
    torch.cuda.synchronize()


def _one_forward(model, x):
    with torch.inference_mode():
        return model(x)


def _power_series(handle, stop, samples):
    import pynvml
    while not stop.is_set():
        try:
            samples.append((time.perf_counter(),
                            pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0))
        except pynvml.NVMLError:
            break
        stop.wait(0.02)


def _energy_per_forward(model, x, repetitions, handle):
    if handle is None:
        return float("nan")
    samples = []
    stop = threading.Event()
    sampler = threading.Thread(target=_power_series, args=(handle, stop, samples),
                               daemon=True)
    _synchronize()
    sampler.start()
    start = time.perf_counter()
    for _ in range(repetitions):
        _one_forward(model, x)
    _synchronize()
    elapsed = time.perf_counter() - start
    stop.set()
    sampler.join()
    if len(samples) < 2:
        return float("nan")
    # Sensor samples cover the whole-GPU power during repeated forwards.
    return float(np.mean([watts for _, watts in samples]) * elapsed / repetitions)


def measure_one(model, image_size, batch, handle=None, latency_repeats=7):
    """Median synchronized wall time, peak PyTorch allocated bytes, GPU joules.

    Input allocation is included in peak, as the input stays alive throughout
    forward. The returned output is released before the memory reading.
    """
    x = torch.randn(batch, 3, image_size, image_size, device="cuda", dtype=torch.float32)
    try:
        for _ in range(3):
            _one_forward(model, x)
        _synchronize()
        torch.cuda.reset_peak_memory_stats()
        times = []
        for _ in range(latency_repeats):
            _synchronize()
            start = time.perf_counter()
            _one_forward(model, x)
            _synchronize()
            times.append(time.perf_counter() - start)
        peak = torch.cuda.max_memory_allocated()
        median = float(np.median(times))
        energy_repeats = max(5, min(1000, math.ceil(0.5 / median)))
        joules = _energy_per_forward(model, x, energy_repeats, handle)
        # Profiler reports estimated conv/linear FLOPs, not a hardware counter.
        with torch.profiler.profile(
            activities=[torch.profiler.ProfilerActivity.CPU],
            with_flops=True, record_shapes=True,
        ) as prof:
            _one_forward(model, x)
            _synchronize()
        prof_flops = sum(event.flops for event in prof.key_averages())
        return {"status": "ok", "latency_s": median,
                "memory_bytes": int(peak), "energy_j": joules,
                "flops_profiler": int(prof_flops),
                "energy_repeats": energy_repeats}
    finally:
        del x


def init_nvml():
    try:
        import pynvml
        pynvml.nvmlInit()
        return pynvml.nvmlDeviceGetHandleByIndex(torch.cuda.current_device())
    except Exception as exc:
        print(f"NVML unavailable: {exc}; energy_j will be NaN")
        return None


def idle_power_watts(handle, samples=20):
    if handle is None:
        return float("nan")
    import pynvml
    _synchronize()
    values = []
    for _ in range(samples):
        values.append(pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0)
        time.sleep(0.025)
    return float(np.median(values))


def measure_grid(path, seed=2026, resume=True):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required for real measurements")
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    device_name = torch.cuda.get_device_name(0)
    model = SmallCNN().cuda().eval()
    handle = init_nvml()
    idle = idle_power_watts(handle)
    grid = build_grid(seed)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    done = set()
    if resume and path.exists() and path.stat().st_size:
        rows = pd.read_csv(path).to_dict("records")
        prior_devices = {str(r.get("gpu_name")) for r in rows}
        if prior_devices != {device_name}:
            raise ValueError(f"Existing CSV is from {prior_devices}; current GPU is {device_name}")
        done = {(int(r["S"]), int(r["B"])) for r in rows}
    for index, row in grid.iterrows():
        s, b = int(row.S), int(row.B)
        if (s, b) in done:
            continue
        record = {"S": s, "B": b, "is_validation": bool(row.is_validation),
                  "gpu_name": device_name, "idle_watts": idle}
        was_oom = False
        try:
            record.update(measure_one(model, s, b, handle))
        except (torch.cuda.OutOfMemoryError, ACCELERATOR_ERROR) as exc:
            if "out of memory" not in str(exc).lower():
                raise
            record.update(status="OOM", latency_s=float("nan"),
                          memory_bytes=float("nan"), energy_j=float("nan"),
                          flops_profiler=float("nan"), energy_repeats=0)
            was_oom = True
        if was_oom:
            gc.collect()
            torch.cuda.empty_cache()
        rows.append(record)
        pd.DataFrame(rows).to_csv(path, index=False)
        if (index + 1) % 10 == 0 or record["status"] == "OOM":
            print(f"{index+1}/{len(grid)}: S={s}, B={b}, {record['status']}")
    return pd.DataFrame(rows).sort_values(["S", "B"]).reset_index(drop=True)


In [ ]:
"""Fit latency and whole-GPU energy parameters on train configurations only."""
import numpy as np
from scipy.optimize import least_squares, nnls



def _training(df, column):
    train = df[(df["status"] == "ok") & (~df["is_validation"])].copy()
    train = train[np.isfinite(train[column]) & (train[column] > 0)]
    if len(train) < 6:
        raise ValueError(f"Need at least 6 finite train measurements for {column}")
    return train


def _fit_latency_model(train, threshold=None):
    f = np.asarray(flops(train.S.to_numpy(), train.B.to_numpy()) / 1e9)
    m = np.asarray(bytes_moved(train.S.to_numpy(), train.B.to_numpy()) / 1e9)
    observed = train.latency_s.to_numpy(dtype=float)
    scale = np.maximum(observed, np.median(observed) * 0.05)
    large = train.B.to_numpy() > threshold if threshold is not None else None

    def residual(p):
        compute = f * (p[1] + p[3] * large) if large is not None else f * p[1]
        return (p[0] + np.maximum(compute, p[2] * m) - observed) / scale

    base_a = max(np.median(observed / np.maximum(f, 1e-12)), 1e-8)
    base_c = max(np.median(observed / np.maximum(m, 1e-12)), 1e-8)
    fits = []
    for af in (0.1, 1.0, 10.0):
        for cf in (0.1, 1.0, 10.0):
            initial = [max(np.min(observed) * 0.5, 1e-6), base_a * af, base_c * cf]
            if large is not None:
                initial.append(base_a)
            fit = least_squares(residual, initial, bounds=(0, np.inf),
                                x_scale="jac", max_nfev=2000)
            if fit.success:
                fits.append(fit)
    if not fits:
        raise RuntimeError("Latency fit failed for all initializations")
    fit = min(fits, key=lambda candidate: np.sum(candidate.fun**2))
    t0, a, c = fit.x[:3]
    theta = {"launch_seconds": float(t0),
             "compute_flops_per_s": float(1e9 / max(a, 1e-15)),
             "memory_bytes_per_s": float(1e9 / max(c, 1e-15))}
    if threshold is not None:
        theta.update(batch_threshold=int(threshold),
                     large_batch_compute_flops_per_s=float(1e9 / max(a + fit.x[3], 1e-15)))
    return theta


def fit_latency(df):
    """Select one/two compute regimes by five-fold CV entirely inside train.

    Candidate thresholds are observed train batch sizes with >=20% of train
    on either side. A second regime must reduce CV mean APE by at least 5%
    relative to the baseline; the selected model is then fitted on all train.
    No validation coordinates, targets or statistics enter model selection.
    """
    train = _training(df, "latency_s").sort_values(["S", "B"]).reset_index(drop=True)
    thresholds = [None] + [int(b) for b in sorted(train.B.unique())
                           if 0.2 <= np.mean(train.B > b) <= 0.8]
    folds = np.random.default_rng(2026).permutation(len(train)) % 5
    scores = []
    for threshold in thresholds:
        predicted = np.empty(len(train))
        for fold in range(5):
            mask = folds == fold
            theta = _fit_latency_model(train.loc[~mask], threshold)
            predicted[mask] = latency(train.loc[mask, "S"].to_numpy(),
                                      train.loc[mask, "B"].to_numpy(), theta)
        scores.append(error_summary(train.latency_s, predicted))
    best = min(range(len(scores)), key=lambda i: scores[i]["mean_ape_percent"])
    if scores[best]["mean_ape_percent"] >= 0.95 * scores[0]["mean_ape_percent"]:
        best = 0
    theta = _fit_latency_model(train, thresholds[best])
    theta["selection"] = {"method": "5-fold train-only CV; mean APE; 5% relative improvement required",
                          "seed": 2026, "n_train": len(train),
                          "candidates": [dict(batch_threshold=t, **score)
                                         for t, score in zip(thresholds, scores)]}
    return theta


def fit_energy(df, theta_latency):
    train = _training(df, "energy_j")
    idle = float(np.median(train.idle_watts.to_numpy(dtype=float)))
    if not np.isfinite(idle):
        raise ValueError("NVML idle power is unavailable; cannot fit energy")
    s, b = train.S.to_numpy(), train.B.to_numpy()
    predicted_time = latency(s, b, theta_latency)
    x = np.column_stack((np.ones(len(train)), flops(s, b) / 1e9,
                         bytes_moved(s, b) / 1e9, predicted_time))
    target = train.energy_j.to_numpy(dtype=float) - idle * predicted_time
    # Weight by observed energy so large, slow configurations do not erase
    # relative accuracy on short forwards. The floor is train-only.
    scale = np.maximum(train.energy_j.to_numpy(dtype=float),
                       np.quantile(train.energy_j, 0.05))
    (fixed, per_gflop, per_gbyte, active), _ = nnls(x / scale[:, None], target / scale)
    return {"latency": theta_latency, "idle_watts": idle,
            "active_watts": float(active), "fixed_joules": float(fixed),
            "joules_per_gflop": float(per_gflop),
            "joules_per_gbyte": float(per_gbyte)}


def error_summary(measured, predicted):
    measured = np.asarray(measured, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = np.isfinite(measured) & np.isfinite(predicted) & (measured > 0)
    if not np.any(valid):
        return {"n": 0, "median_ape_percent": None, "mean_ape_percent": None}
    ape = 100 * np.abs(predicted[valid] - measured[valid]) / measured[valid]
    return {"n": int(valid.sum()), "median_ape_percent": float(np.median(ape)),
            "mean_ape_percent": float(np.mean(ape))}


## 1. Модель и соглашения

Вход: `B × 3 × S × S`, где `S` кратен 16. Шесть блоков `Conv → BatchNorm → ReLU(inplace)`, после первого блока `MaxPool`. Затем `GlobalAvgPool → Linear(512,256) → ReLU(inplace) → Linear(256,100)`. Каждая `Conv` имеет `bias=False`, `padding=k//2`; сеть работает в `eval()`, FP32 и `torch.inference_mode()`.

| i | Conv | Выходные каналы | Spatial size |
|---|---|---:|---:|
| 1 | 7×7, stride 2, 3→32 | 32 | S/2, после MaxPool — S/4 |
| 2 | 5×5, 32→64 | 64 | S/4 |
| 3 | 3×3, stride 2, 64→128 | 128 | S/8 |
| 4 | 1×1, 128→256 | 256 | S/8 |
| 5 | 3×3, stride 2, 256→256 | 256 | S/16 |
| 6 | 1×1, 256→512 | 512 | S/16 |

In [ ]:
model_cpu = SmallCNN().eval()
with torch.inference_mode():
    assert model_cpu(torch.randn(2, 3, 32, 32)).shape == (2, 100)
print(model_cpu)
print(f'Параметры + buffers: {model_bytes()/2**20:.3f} MiB')

## 2. Вывод аналитических уравнений

Обозначим для каждой свёртки `(c_in,i, c_out,i, k_i, r_i)`, где `r_i` — spatial size из таблицы. `W` — байты параметров и buffers сети; `Q(S,B)` — модельный объём передачи данных из `bytes_moved()`.

**FLOPs:** один MAC = 2 FLOPs. `BatchNorm` в `eval()` считаем как scale + shift (2 FLOPs на output), `GlobalAvgPool` — `(n−1)` сложений + 1 деление (всего `n`), для Linear добавляем bias. ReLU и MaxPool используют сравнения и в floating-point operations не входят:

\[
F(S,B)=B\sum_{i=1}^{6}c_{out,i}r_i^2(2c_{in,i}k_i^2+2)
 +512B(S/16)^2+B(2\cdot512\cdot256+256+2\cdot256\cdot100+100).
\]

**Peak memory:** метрика — `torch.cuda.max_memory_allocated()` после прогрева, включая живые allocations библиотек. Старая оценка `M_live=W+76BS²` пропускала постоянный workspace cuBLAS. Исправленная формула для T4, native allocator, одного stream и default workspace:

\[
M(S,B)=W_{alloc}+W_{BLAS}+76BS^2,\qquad
W_{alloc}=\sum_j512\left\lceil\frac{n_j}{512}\right\rceil=4\,187\,136,
\]
\[
W_{BLAS}=(2\cdot4096+8\cdot16)\cdot1024=8\,519\,680\ \text{bytes}.
\]

Здесь `n_j` — байты отдельного параметра/buffer (включая шесть int64 BN counters); размер каждого выделения округляется до 512 bytes. `76BS²=12BS²+2·32BS²` — постоянный вход и две активации первого BatchNorm. При `S` кратном 16 эти три размера уже кратны 512. `W` без округления равен 4 181 312 bytes и по-прежнему используется в сравнении raw parameter storage.

`W_BLAS` задан в [исходниках PyTorch 2.11](https://github.com/pytorch/pytorch/blob/v2.11.0/aten/src/ATen/cuda/CublasHandlePool.cpp), округление — в [CUDA allocator](https://github.com/pytorch/pytorch/blob/v2.11.0/c10/cuda/CUDACachingAllocator.cpp). **Ни один коэффициент памяти не подгоняется по CSV.** После прогрева Linear этот workspace остаётся выделенным и входит в измеряемый peak. Формула предполагает отсутствие override `CUBLAS_WORKSPACE_CONFIG`, дополнительных streams и посторонних GPU tensors.

Дополнительные временные workspace cuDNN, отдельные пулы cuBLASLt и особенности повторного использования blocks всё ещё не учтены. Поэтому остаётся недооценка, особенно для отдельных batch/shape. CUDA context и свободные reserved blocks не входят в `max_memory_allocated()` и не добавляются к формуле.

**Bytes moved:** для каждой Conv учитываем один read input, один read weights и один write output; для BN/ReLU — read+write; для MaxPool/GAP/Linear аналогично. Игнорируем cache reuse, fusion и cuDNN workspace. Подробная сумма находится в `equations.py`; она не подгоняется по GPU.

**Latency** (seconds):
\[
T(S,B;\theta)=t_0+\max(F(S,B)/P(B),\ Q(S,B)/R),
\quad P(B)=\begin{cases}P_1,& B\le B_*\\P_2,& B>B_*\end{cases}.
\]
`launch-bound`, `memory-bound`, `compute-bound` — соответственно преобладание `t0`, `Q/R`, `F/P(B)`. Порог `B*` и выбор одного/двух compute regimes определяются 5-fold CV только на train по mean APE. Второй режим принимается при улучшении минимум на 5%; затем коэффициенты подгоняются на всём train. Ограничение `P2 ≤ P1` описывает наблюдаемое замедление больших batch. Это эффективные скорости, не аппаратные константы; причина смены режима требует отдельного GPU-профилирования.

**Energy** (joules, вся GPU):
\[
E(S,B;\theta_E)=(P_{idle}+P_{active})T(S,B;\theta)+e_0+e_FF(S,B)/10^9+e_QQ(S,B)/10^9.
\]
`P_idle` измеряем через NVML. `P_active` — дополнительная мощность GPU во время работы; все остальные коэффициенты неотрицательно подгоняем только по train points. Во время измерений мощность включает и idle часть GPU.

In [ ]:
S_demo = np.array([32, 128, 512])[:, None]
B_demo = np.array([1, 16, 256])[None, :]
print('FLOPs, broadcast shape:', flops(S_demo, B_demo).shape)
print('Memory [GiB]:\n', np.round(memory(S_demo, B_demo) / 2**30, 3))
print('Bytes moved [GB]:\n', np.round(bytes_moved(S_demo, B_demo) / 1e9, 3))

## 3. Measurement grid и holdout

Семь заданных `S` + четыре случайных кратных 16; девять заданных `B` + три случайных неповторяющихся целых дают 132 пары. Фиксированный seed заранее выбирает 20% пар в `validation`; эти измерения не используются для calibration. Все inputs — случайные тензоры.

In [ ]:
grid = build_grid(seed=2026)
assert len(grid) == 132 and not grid.duplicated(['S', 'B']).any()
print('S:', sorted(grid.S.unique()))
print('B:', sorted(grid.B.unique()))
print('train:', (~grid.is_validation).sum(), 'validation:', grid.is_validation.sum())
display(grid.head(12))

## 4. Реальные GPU measurements

Протокол фиксируется в `measure.py`: `cudnn.benchmark=False`, оба `allow_tf32=False`, `eval()`, FP32 и `inference_mode()`. Для каждой пары: warmup, медиана из семи synchronized wall-clock forwards, `max_memory_allocated`, оценка FLOPs через PyTorch profiler и NVML power за серию повторов для энергии одного forward. Profiler даёт **оценку FLOPs операторов Conv/Linear**, а не счётчик инструкций GPU; его расхождение с аналитическим `F` включает BN/GAP.

При `torch.cuda.OutOfMemoryError` строка получает `status=OOM`; CSV сохраняется после каждой точки. NVML должен быть доступен для калибровки энергии. В Colab при необходимости выполните `%pip install nvidia-ml-py scipy pandas matplotlib` и перезапустите kernel.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Для полного grid нужна CUDA GPU')
measurements_path = RESULTS / 'measurements.csv'
df = measure_grid(measurements_path, seed=2026, resume=True)
print(df.status.value_counts())
print('NVML energy points:', np.isfinite(df.energy_j).sum())
display(df.head())

## 5. Calibration без leakage

Коэффициенты, кандидаты порога и выбор модели используют только `status=ok`, `is_validation=False`. Latency: baseline `t0 + max(F/P,Q/R)` сравнивается с двумя compute rates через 5-fold CV внутри train (mean APE, минимум 5% относительного улучшения). Energy: неотрицательная подгонка idle + active power и стоимости FLOPs/traffic. Validation не участвует в fit; поскольку прежний отчёт уже показывал этот holdout, независимое подтверждение новой гипотезы требует свежих измерений.


In [ ]:
assert len(df) == 132, 'Grid ещё не завершён; повторно запустите ячейку измерения'
theta_t = fit_latency(df)
theta_e = fit_energy(df, theta_t)
theta = {
    'latency': theta_t,
    'energy': theta_e,
    'software': {'python': platform.python_version(), 'torch': torch.__version__,
                 'cuda': torch.version.cuda},
    'gpu_name': torch.cuda.get_device_name(0),
    'grid_seed': 2026,
    'solution_version': '2-memory-refinement',
    'run_id': RUN_ID,
    'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
}
(RESULTS / 'theta.json').write_text(json.dumps(theta, ensure_ascii=False, indent=2))
print(json.dumps(theta, ensure_ascii=False, indent=2))

## 6. Validation: measured points и predicted curves

Графики строятся без подгонки на `validation`. Оси имеют единицы; линии — аналитическое предсказание, маркеры — реальные значения (`○ train`, `◆ validation`). Для памяти `× OOM` ставится на уровне VRAM capacity. Графики текущего прогона сохраняются в `results_v2/runs/<RUN_ID>/figures/`; графики опубликованного прогона находятся в `results_v2/figures/`.

In [ ]:
"""Recalibrate saved measurements and regenerate reports without a CUDA run.

Usage: python -m hw1.report --keep-calibration
Defaults to results_v2; version 1 uses its own embedded notebook code.
The software metadata in theta.json describes the original measurements.
"""
import argparse
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy



def write_report(df, results, theta, baseline=None):
    results = Path(results)
    (results / 'figures').mkdir(parents=True, exist_ok=True)
    df = df.copy()
    df['flops_pred'] = flops(df.S, df.B)
    df['memory_pred'] = memory(df.S, df.B)
    df['latency_pred'] = latency(df.S, df.B, theta['latency'])
    df['energy_pred'] = energy(df.S, df.B, theta['energy'])
    metrics = [('flops_profiler', 'flops_pred', 'FLOPs', 1e9, 'GFLOPs'),
               ('memory_bytes', 'memory_pred', 'Peak memory', 2**30, 'GiB'),
               ('latency_s', 'latency_pred', 'Latency', 1e-3, 'ms'),
               ('energy_j', 'energy_pred', 'Energy', 1., 'J')]
    scores = {}
    theta = dict(theta, memory_model='live tensors + 512-byte model allocation rounding + default pre-Hopper cuBLAS workspace; no fit')
    for split, flag in [('train', False), ('validation', True)]:
        part = df[df.status.eq('ok') & df.is_validation.eq(flag)]
        scores[split] = {title: error_summary(part[measured], part[predicted])
                        for measured, predicted, title, _, _ in metrics}
    valid = df[df.status.eq('ok') & df.is_validation]
    scores['memory_comparison_validation'] = {
        'live_tensors_only': error_summary(valid.memory_bytes, memory_live_tensors(valid.S, valid.B)),
        'with_runtime_workspace': scores['validation']['Peak memory']}
    if baseline is not None:
        part = df[df.status.eq('ok') & df.is_validation]
        scores['baseline_validation'] = {
            'Latency': error_summary(part.latency_s, latency(part.S, part.B, baseline['latency'])),
            'Energy': error_summary(part.energy_j, energy(part.S, part.B, baseline['energy']))}
    good = df[df.status.eq('ok')]
    terms = latency_terms(good.S, good.B, theta['latency'])
    regimes = pd.Series(np.array(['launch-bound', 'memory-bound', 'compute-bound'])[
        np.argmax(np.column_stack(terms), axis=1)]).value_counts().to_dict()
    capacity = theta.get('gpu_memory_bytes')
    actual_oom = df.status.eq('OOM').to_numpy()
    scores['oom'] = {'actual': int(actual_oom.sum()), 'predicted': None}
    if capacity is not None:
        predicted_oom = df.memory_pred.to_numpy() > capacity
        scores['oom'].update(predicted=int(predicted_oom.sum()),
                             false_positive=int((predicted_oom & ~actual_oom).sum()),
                             false_negative=int((~predicted_oom & actual_oom).sum()))
    for measured, predicted, title, factor, unit in metrics:
        fig, axes = plt.subplots(3, 4, figsize=(20, 14))
        for ax, s in zip(axes.flat, sorted(df.S.unique())):
            sub = df[df.S.eq(s)].sort_values('B')
            ax.plot(sub.B, sub[predicted] / factor, label='prediction')
            for flag, marker, label in [(False, 'o', 'measured train'), (True, 'D', 'measured validation')]:
                points = sub[sub.status.eq('ok') & sub.is_validation.eq(flag)]
                ax.scatter(points.B, points[measured] / factor, marker=marker, label=label)
            if measured == 'memory_bytes' and capacity is not None:
                ax.axhline(capacity / factor, color='gray', linestyle=':', label='GPU capacity')
                oom = sub[sub.status.eq('OOM')]
                ax.scatter(oom.B, np.full(len(oom), capacity / factor), marker='x', label='OOM')
            ax.set(xlabel='Batch size B', ylabel=f'{title} [{unit}]', title=f'S={s}')
            ax.set_xscale('log', base=2)
            ax.grid(alpha=0.25)
        for ax in list(axes.flat)[df.S.nunique():]:
            ax.axis('off')
        handles, labels = axes.flat[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', ncol=5)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        fig.savefig(results / 'figures' / f'{measured}.png', dpi=160)
        plt.close(fig)
    t = theta['latency']
    summary = ['# Результаты на T4',
               f"GPU: {theta['gpu_name']}; measurement software: {theta['software']}.",
               f'Grid: {len(df)}; ok={len(good)}; OOM={int(df.status.eq("OOM").sum())}.',
               f'Train={int((~df.is_validation).sum())}; validation={int(df.is_validation.sum())}.',
               '', '## Validation errors', '',
               '| Metric | Median APE, % | Mean APE, % |', '|---|---:|---:|']
    for title, score in scores['validation'].items():
        summary.append(f'| {title} | {score["median_ape_percent"]:.2f} | {score["mean_ape_percent"]:.2f} |')
    before_memory = scores['memory_comparison_validation']['live_tensors_only']
    summary += ['', 'Память без подгонки: median APE '
                f'{before_memory["median_ape_percent"]:.2f}% → '
                f'{scores["validation"]["Peak memory"]["median_ape_percent"]:.2f}%.']
    if baseline is not None:
        summary += ['', 'Сравнение с исходными параметрами (тот же CSV и holdout):']
        for title, score in scores['baseline_validation'].items():
            summary.append(f'- {title}: median APE {score["median_ape_percent"]:.2f}% → '
                           f'{scores["validation"][title]["median_ape_percent"]:.2f}%.')
    summary += ['', '## Калибровка и ограничения',
                f'- t0={t["launch_seconds"]:.7g} s; P={t["compute_flops_per_s"]:.7g} FLOP/s; R={t["memory_bytes_per_s"]:.7g} byte/s.',
                f'- Batch threshold={t.get("batch_threshold")}; large-batch P={t.get("large_batch_compute_flops_per_s")} FLOP/s.',
                '- Порог и выбор модели: 5-fold CV только внутри train, критерий mean APE; минимум 5% относительного улучшения. Подробности в theta.json.',
                f'- Idle={theta["energy"]["idle_watts"]:.3f} W; дополнительная active power={theta["energy"]["active_watts"]:.3f} W.',
                f'- Режимы по максимальному члену уравнения: {regimes}. Это классификация модели, не аппаратный профиль.',
                '- FLOPs включает BN/GAP; profiler — Conv/Linear. Память: округлённые веса/buffers + живые тензоры + постоянный cuBLAS workspace 8,125 MiB. Коэффициенты не подгонялись.',
                '- Допущения памяти: T4/pre-Hopper, native allocator, один stream, без CUBLAS_WORKSPACE_CONFIG override и посторонних CUDA tensors. Дополнительные cuDNN/cuBLASLt workspace остаются вне формулы.',
                '- Размер cuBLAS взят из [исходников PyTorch 2.11](https://github.com/pytorch/pytorch/blob/v2.11.0/aten/src/ATen/cuda/CublasHandlePool.cpp); 512-byte rounding — из [CUDA allocator](https://github.com/pytorch/pytorch/blob/v2.11.0/c10/cuda/CUDACachingAllocator.cpp).',
                '- Два эффективных compute rate описывают наблюдаемый batch-режим. Причина (kernel selection, clocks и т.п.) не установлена; граница между соседними измеренными B не локализована точно.',
                '- R и энергетические коэффициенты могут быть плохо идентифицируемы. Это эффективные параметры всей сети; не физические спецификации GPU.',
                '- NVML energy измеряется за серию forwards, latency — синхронными одиночными вызовами; это добавляет систематическую погрешность.',
                '- Holdout уже использовался в предыдущем отчёте. Коэффициенты и порог не подгонялись по нему, но для независимой проверки новой гипотезы нужны новые измерения.',
                f'- OOM: {scores["oom"]}. ' + ('Фактических OOM нет; точность границы OOM не проверена.' if not actual_oom.any() else 'Сопоставьте ошибки с workspace и доступной VRAM.')]
    if capacity is None:
        summary.append('- Точная ёмкость исходной GPU не сохранена; при offline-пересчёте граница OOM и линия capacity не восстанавливаются по другой GPU.')
    (results / 'theta.json').write_text(json.dumps(theta, ensure_ascii=False, indent=2) + '\n')
    (results / 'errors.json').write_text(json.dumps(scores, ensure_ascii=False, indent=2) + '\n')
    (results / 'results_summary.md').write_text('\n'.join(summary) + '\n')
    return scores



scores = write_report(df, RESULTS, theta)
from IPython.display import Image, display
for metric in ['flops_profiler', 'memory_bytes', 'latency_s', 'energy_j']:
    display(Image(filename=str(RESULTS / 'figures' / f'{metric}.png')))


## 7. Ошибки и обсуждение

`MAPE` вычисляем отдельно для train и unseen validation. Для `OOM` сравниваем `Memory(S,B)` с физической ёмкостью GPU, но формула учитывает только постоянный cuBLAS workspace, без временных workspace cuDNN. В отчёте обсуждаем: (1) где доминирует запуск kernels, (2) где растёт traffic, (3) где ограничивает compute, (4) какие предсказания не работают и почему. Отдельно укажите шум NVML и то, что profiler FLOPs — оценка Conv/Linear.

In [ ]:
print(json.dumps(scores, ensure_ascii=False, indent=2))


In [ ]:
print((RESULTS / 'results_summary.md').read_text())


## 8. Файлы для представления работы

В одной папке `hw1/`: первая версия — `hw1.ipynb` и `results_v1/`; вторая — `hw1_version_2.ipynb`, актуальные `.py`, тесты и `results_v2/`. README объясняет различия и ограничения. В архиве нового прогона находятся CSV, theta, ошибки, сводка и графики; после проверки их можно перенести в корень `results_v2/`.

Рукописные выводы сохранены отдельно: `hw1_handwritten.pdf` для v1 и `hw1_handwritten_v2.pdf` для v2. Вторая версия включает `P(B)`, член `P_active·T`, округление allocations и постоянный workspace cuBLAS. Печатные пояснения доступны в `handwritten_v2_guide.pdf`; актуальные формулы также даны в разделе 2 и `derivations_v2.md`.


In [ ]:
import shutil
archive = shutil.make_archive(str(RESULTS.parent / f'hw1_version_2_results_{RUN_ID}'), 'zip', root_dir=RESULTS)
print('Архив нового прогона:', archive)
try:
    from google.colab import files
except ImportError:
    print('Скачайте архив через файловую панель Jupyter.')
else:
    files.download(archive)
